# Data Cleaning

In [9]:
import pandas as pd
import os
import ast
from pathlib import Path

In [19]:
df = pd.read_csv("../data/raw/anime_data.csv")
current_df = pd.read_csv("../data/raw/current_data.csv")

Let's remove duplicates.

In [20]:
df2 = df.drop_duplicates(subset=['mal_id'], keep='first')
current_df2 = current_df.drop_duplicates(subset=['mal_id'], keep='first')

## Future Anime

In the data collection process, one of the criteria was that it should not be currently airing. This will of course include Fall 2026 and future anime. We will remove anime with release year 2027 and above, since we need Fall 2026 anime as our final prediction data.

In [21]:
df3 = df2[df2['year'] < 2027]

current_df3 = current_df # just for tracking purposes

df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 5308 entries, 0 to 5307
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mal_id                 5308 non-null   int64  
 1   title                  5308 non-null   str    
 2   source                 5308 non-null   str    
 3   episodes               5308 non-null   int64  
 4   synopsis               5272 non-null   str    
 5   year                   5308 non-null   int64  
 6   season                 5308 non-null   str    
 7   producers              5308 non-null   str    
 8   genres                 5308 non-null   str    
 9   studios                5308 non-null   str    
 10  demographics           5308 non-null   str    
 11  themes                 5308 non-null   str    
 12  rating                 5278 non-null   str    
 13  sequel                 5308 non-null   bool   
 14  favorites              5308 non-null   int64  
 15  score          

## Synopsis

No synopsis should be fine. My justification is that shows with no synopsis might be boring for users who look at MAL. However, categorizing by age rating is important: we need to know if restrictive shows have lower metrics.

In [22]:
df4 = df3[~df3['rating'].isna()]
current_df4 = current_df3

df4.info()

<class 'pandas.DataFrame'>
Index: 5278 entries, 0 to 5307
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mal_id                 5278 non-null   int64  
 1   title                  5278 non-null   str    
 2   source                 5278 non-null   str    
 3   episodes               5278 non-null   int64  
 4   synopsis               5243 non-null   str    
 5   year                   5278 non-null   int64  
 6   season                 5278 non-null   str    
 7   producers              5278 non-null   str    
 8   genres                 5278 non-null   str    
 9   studios                5278 non-null   str    
 10  demographics           5278 non-null   str    
 11  themes                 5278 non-null   str    
 12  rating                 5278 non-null   str    
 13  sequel                 5278 non-null   bool   
 14  favorites              5278 non-null   int64  
 15  score               

Additionally, we can fill the null synopsis entries with an empty string for data entry purposes.

In [24]:
df4.fillna({'synopsis': " "}, inplace=True)
current_df4.fillna({'synopsis': " "}, inplace=True)

df4.info()

<class 'pandas.DataFrame'>
Index: 5278 entries, 0 to 5307
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mal_id                 5278 non-null   int64  
 1   title                  5278 non-null   str    
 2   source                 5278 non-null   str    
 3   episodes               5278 non-null   int64  
 4   synopsis               5278 non-null   str    
 5   year                   5278 non-null   int64  
 6   season                 5278 non-null   str    
 7   producers              5278 non-null   str    
 8   genres                 5278 non-null   str    
 9   studios                5278 non-null   str    
 10  demographics           5278 non-null   str    
 11  themes                 5278 non-null   str    
 12  rating                 5278 non-null   str    
 13  sequel                 5278 non-null   bool   
 14  favorites              5278 non-null   int64  
 15  score               

## Multi-valued Features

If we check some multi-valued features such as genres, we can see that they're not actually lists, but strings.

In [ ]:
df4['genres']

0                ['Action', 'Award Winning', 'Sci-Fi']
1                    ['Action', 'Adventure', 'Sci-Fi']
2       ['Action', 'Drama', 'Mystery', 'Supernatural']
3                   ['Action', 'Adventure', 'Fantasy']
4                                           ['Sports']
                             ...                      
8719                 ['Fantasy', 'Gourmet', 'Romance']
8726                                       ['Romance']
8735                ['Action', 'Adventure', 'Fantasy']
8741                             ['Comedy', 'Romance']
8816                        ['Comedy', 'Supernatural']
Name: genres, Length: 5338, dtype: str

Let's turn them into actual lists.

In [25]:
def parse_list_col(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):  # already a real list, don't double-parse
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

df4[['themes', 'genres', 'studios', 'demographics', 'producers']] = df4[['themes', 'genres', 'studios', 'demographics', 'producers']].map(parse_list_col)
current_df4[['themes', 'genres', 'studios', 'demographics', 'producers']] = current_df4[['themes', 'genres', 'studios', 'demographics', 'producers']].map(parse_list_col)
current_df4['genres']

0                 [Drama, Mystery]
1                [Action, Fantasy]
2     [Action, Adventure, Fantasy]
3                [Romance, Sports]
4                  [Action, Drama]
                  ...             
67                        [Comedy]
68                              []
69        [Fantasy, Slice of Life]
70                        [Comedy]
71                       [Fantasy]
Name: genres, Length: 72, dtype: object

For now, we can save this.

In [27]:
target_dir = Path("../data/processed")
file_path1 = target_dir / "anime_data_1.parquet"
df4.to_parquet(file_path1, engine='pyarrow')

file_path2 = target_dir / "real_data_1.parquet"
current_df4.to_parquet(file_path2, engine='pyarrow')